In [1]:
from PIL import Image
from ps3 import PS3VisionModel, PS3ImageProcessor

from transformers import (
    AutoProcessor,
    SiglipImageProcessor,
    SiglipVisionModel)
import torch

/root/miniconda3/envs/univa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/univa/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:

# Load the PS3 model and processor.
vision_model = PS3VisionModel.from_pretrained("nvidia/PS3-1.5K-SigLIP2")
processor = PS3ImageProcessor.from_pretrained("nvidia/PS3-1.5K-SigLIP2")
vision_model.cuda().eval()


SIGLIP_PATH="/workspace/cleanroom/model_weights/siglip2-so400m-patch16-512"

siglip_processor = SiglipImageProcessor.from_pretrained(SIGLIP_PATH)
siglip_model = SiglipVisionModel.from_pretrained(
    SIGLIP_PATH,
    torch_dtype=torch.bfloat16,
).to("cuda")

/root/miniconda3/envs/univa/lib/python3.10/site-packages/ps3/modeling_ps3.py:142: UserWarning: The number of hidden layers to return hidden states is currently set to 27. If this value is large, it can consume a lot of memory. Consider setting it to a smaller value if you won't use all the hidden states from every layer!
  warnings.warn(f"The number of hidden layers to return hidden states is currently set to {self.num_hidden_layers_to_return}. If this value is large, it can consume a lot of memory. Consider setting it to a smaller value if you won't use all the hidden states from every layer!")
Some weights of the model checkpoint at nvidia/PS3-1.5K-SigLIP2 were not used when initializing PS3VisionModel: ['logit_bias', 'logit_scale', 'text_model.ln_final.bias', 'text_model.ln_final.weight', 'text_model.positional_embedding', 'text_model.prompt_proj.fc1.bias', 'text_model.prompt_proj.fc1.weight', 'text_model.prompt_proj.fc2.bias', 'text_model.prompt_proj.fc2.weight', 'text_model.prompt

In [19]:

# # You can replace it with your own image.
image = Image.open("/workspace/UniWorld-V1/assets/removal_3.jpg").resize((1024, 1024))

# Preprocess the image.
x = processor(image)["pixel_values"][0].unsqueeze(0).cuda()
outs = vision_model(x, num_look_close=1)
features = outs.last_hidden_state #  (378/14)^2 = 729, (756/14)^2 = 2916, (1512/14)^2 = 11664 -> 15309
print(features.shape)  # (1, 88209, 1152)


torch.Size([1, 3289, 1152])


In [15]:

tensors = siglip_processor.preprocess(
        image.convert("RGB"),
        do_resize=True,
        do_convert_rgb=True,
        return_tensors="pt",
    ).pixel_values.cuda()

        
        
siglip_hs = siglip_model(tensors).last_hidden_state

print(siglip_hs.shape)

torch.Size([1, 1024, 1152])


torch.Size([1, 3, 512, 512])